# Huấn luyện mô hình đánh giá độ phù hợp giữa CV và JD

Notebook này được viết để chạy trực tiếp trên Google Colab.

Mục tiêu:

1. Kiểm tra CV và JD bị trùng trước khi train.
2. Chia train và validation theo nhóm, tránh cùng CV hoặc JD xuất hiện ở cả hai tập.
3. So sánh nhiều cách cắt văn bản và nhiều loại loss.
4. Chỉ dùng tập test sau khi đã chọn xong cấu hình tốt nhất.
5. Đánh giá bằng Accuracy, Macro F1, F1 từng nhãn và ma trận nhầm lẫn.

**Cách chạy:** chọn `Runtime → Change runtime type → T4 GPU`, sau đó chạy lần lượt từ trên xuống dưới.

## 1 Cài thư viện

Ô này cài các thư viện cần thiết. Sau khi chạy xong, Colab có thể yêu cầu khởi động lại phiên nếu phiên bản thư viện thay đổi lớn.

In [ ]:
!pip install -q -U transformers datasets accelerate scikit-learn pandas numpy matplotlib seaborn sentencepiece

## 2 Khai báo thư viện và cấu hình chung

- `RUN_MODE = "quick"`: chạy ba cấu hình chính để kiểm tra nhanh.
- `RUN_MODE = "full"`: chạy toàn bộ sáu cấu hình.
- `REMOVE_TEST_OVERLAP = True`: xóa khỏi train những dòng có CV hoặc JD trùng với test để kết quả trung thực hơn.

Lần đầu nên chạy `quick`. Khi notebook chạy ổn mới đổi sang `full`.

In [ ]:
import os
import gc
import re
import json
import math
import random
import hashlib
import shutil
from pathlib import Path
from dataclasses import dataclass, asdict

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

import torch
import torch.nn as nn
import torch.nn.functional as F

from datasets import load_dataset
from sklearn.model_selection import GroupShuffleSplit
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score,
    ndcg_score,
)
from transformers import (
    AutoModelForSequenceClassification,
    AutoTokenizer,
    EarlyStoppingCallback,
    Trainer,
    TrainingArguments,
    set_seed as hf_set_seed,
)

os.environ["TOKENIZERS_PARALLELISM"] = "false"

RUN_MODE = "quick"                 # "quick" hoặc "full"
REMOVE_TEST_OVERLAP = True          # Khuyên dùng True
BASE_MODEL = "distilbert-base-uncased"
RANDOM_SEED = 42
MAX_LENGTH = 512
OUTPUT_ROOT = Path("/content/cvjd_optimized_runs")
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)

LABEL2ID = {"No Fit": 0, "Potential Fit": 1, "Good Fit": 2}
ID2LABEL = {0: "No Fit", 1: "Potential Fit", 2: "Good Fit"}
LABEL_NAMES = [ID2LABEL[i] for i in range(3)]

def seed_everything(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    hf_set_seed(seed)

seed_everything(RANDOM_SEED)

print("PyTorch:", torch.__version__)
print("CUDA:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print("VRAM:", round(torch.cuda.get_device_properties(0).total_memory / 1e9, 2), "GB")
else:
    print("CẢNH BÁO: Chưa bật GPU. Hãy chọn T4 GPU trước khi train.")

## 3 Tải dữ liệu

Mỗi dòng dữ liệu gồm:

- `resume_text`: nội dung CV.
- `job_description_text`: nội dung mô tả công việc.
- `label`: No Fit, Potential Fit hoặc Good Fit.

Tập test có sẵn được giữ riêng và không dùng để chọn cấu hình.

In [ ]:
raw_dataset = load_dataset("cnamuangtoun/resume-job-description-fit")

raw_train_df = pd.DataFrame(raw_dataset["train"])
raw_test_df = pd.DataFrame(raw_dataset["test"])

print("Train gốc:", len(raw_train_df))
print("Test gốc:", len(raw_test_df))
print("\nPhân bố nhãn train:")
print(raw_train_df["label"].value_counts())
print("\nPhân bố nhãn test:")
print(raw_test_df["label"].value_counts())

## 4 Làm sạch và kiểm tra dữ liệu trùng

Notebook cũ chỉ kiểm tra cặp CV–JD trùng hoàn toàn. Phiên bản này tạo ba mã:

- `resume_hash`: nhận biết CV giống nhau.
- `jd_hash`: nhận biết JD giống nhau.
- `pair_hash`: nhận biết đúng cặp CV và JD.

Nhãn mâu thuẫn được kiểm tra trước khi bỏ dòng trùng. Nếu cùng một cặp có nhiều nhãn, toàn bộ cặp đó được loại vì không biết nhãn nào đúng.

In [ ]:
def normalize_text(text):
    text = str(text).replace("\u00a0", " ")
    text = re.sub(r"\s+", " ", text).strip().lower()
    return text

def md5_text(text):
    return hashlib.md5(text.encode("utf-8")).hexdigest()

def prepare_dataframe(df, name):
    out = df.copy()
    before = len(out)

    out["resume_clean"] = out["resume_text"].map(normalize_text)
    out["jd_clean"] = out["job_description_text"].map(normalize_text)
    out = out[(out["resume_clean"].str.len() > 50) & (out["jd_clean"].str.len() > 50)].copy()

    out["resume_hash"] = out["resume_clean"].map(md5_text)
    out["jd_hash"] = out["jd_clean"].map(md5_text)
    out["pair_hash"] = (out["resume_hash"] + "|" + out["jd_hash"]).map(md5_text)

    # Kiểm tra mâu thuẫn trước khi xóa dòng trùng.
    conflict_pairs = out.groupby("pair_hash")["label"].nunique()
    conflict_pairs = set(conflict_pairs[conflict_pairs > 1].index)
    conflict_rows = out["pair_hash"].isin(conflict_pairs).sum()
    if conflict_pairs:
        out = out[~out["pair_hash"].isin(conflict_pairs)].copy()

    duplicate_rows = out.duplicated("pair_hash", keep="first").sum()
    out = out.drop_duplicates("pair_hash", keep="first").copy()

    out["label"] = out["label"].map(lambda x: LABEL2ID.get(x, x)).astype(int)
    out = out.reset_index(drop=True)

    print(f"[{name}] {before} -> {len(out)} mẫu")
    print(f"  Dòng trùng cặp đã bỏ: {duplicate_rows}")
    print(f"  Dòng thuộc cặp mâu thuẫn đã bỏ: {conflict_rows}")
    return out

train_all_df = prepare_dataframe(raw_train_df, "Train")
test_df = prepare_dataframe(raw_test_df, "Test")

overlap_pair = set(train_all_df.pair_hash) & set(test_df.pair_hash)
overlap_resume = set(train_all_df.resume_hash) & set(test_df.resume_hash)
overlap_jd = set(train_all_df.jd_hash) & set(test_df.jd_hash)

print("\nTRÙNG GIỮA TRAIN VÀ TEST")
print("Cặp CV-JD trùng:", len(overlap_pair))
print("CV trùng:", len(overlap_resume))
print("JD trùng:", len(overlap_jd))

if REMOVE_TEST_OVERLAP:
    keep_mask = (
        ~train_all_df["resume_hash"].isin(overlap_resume)
        & ~train_all_df["jd_hash"].isin(overlap_jd)
    )
    removed = int((~keep_mask).sum())
    train_all_df = train_all_df[keep_mask].reset_index(drop=True)
    print(f"Đã bỏ {removed} dòng train có CV hoặc JD trùng với test.")
    print("Train còn lại:", len(train_all_df))

if len(train_all_df) < 1000:
    raise ValueError("Train còn quá ít mẫu sau khi loại trùng. Hãy kiểm tra báo cáo overlap trước khi tiếp tục.")

## 5 Chia train và validation theo nhóm

Một CV có thể đi với nhiều JD và một JD có thể đi với nhiều CV. Vì vậy, notebook tạo nhóm liên kết giữa CV và JD. Toàn bộ một nhóm chỉ được nằm ở train hoặc validation.

Cách này giúp validation khó hơn nhưng đáng tin hơn cách chia ngẫu nhiên từng dòng.

In [ ]:
class UnionFind:
    def __init__(self):
        self.parent = {}

    def find(self, x):
        if x not in self.parent:
            self.parent[x] = x
        if self.parent[x] != x:
            self.parent[x] = self.find(self.parent[x])
        return self.parent[x]

    def union(self, a, b):
        ra, rb = self.find(a), self.find(b)
        if ra != rb:
            self.parent[rb] = ra

def add_component_id(df):
    uf = UnionFind()
    for row in df.itertuples(index=False):
        uf.union("R:" + row.resume_hash, "J:" + row.jd_hash)
    result = df.copy()
    result["component_id"] = [
        uf.find("R:" + h) for h in result["resume_hash"]
    ]
    return result

train_all_df = add_component_id(train_all_df)

# Thử nhiều cách chia nhóm và chọn cách có kích thước cùng phân bố nhãn gần mục tiêu nhất.
splitter = GroupShuffleSplit(n_splits=50, test_size=0.15, random_state=RANDOM_SEED)
full_dist = train_all_df["label"].value_counts(normalize=True).sort_index()
best = None

for train_idx, val_idx in splitter.split(train_all_df, groups=train_all_df["component_id"]):
    val_part = train_all_df.iloc[val_idx]
    val_dist = val_part["label"].value_counts(normalize=True).reindex([0, 1, 2], fill_value=0)
    size_error = abs(len(val_part) / len(train_all_df) - 0.15)
    label_error = float((val_dist - full_dist.reindex([0, 1, 2], fill_value=0)).abs().mean())
    score = size_error + label_error
    if best is None or score < best[0]:
        best = (score, train_idx, val_idx)

_, train_idx, val_idx = best
train_df = train_all_df.iloc[train_idx].reset_index(drop=True)
val_df = train_all_df.iloc[val_idx].reset_index(drop=True)

assert not (set(train_df.resume_hash) & set(val_df.resume_hash)), "CV còn trùng giữa train và validation"
assert not (set(train_df.jd_hash) & set(val_df.jd_hash)), "JD còn trùng giữa train và validation"

def show_split(name, df):
    counts = df["label"].value_counts().reindex([0, 1, 2], fill_value=0)
    print(f"\n{name}: {len(df)} mẫu")
    for label_id, count in counts.items():
        print(f"  {ID2LABEL[label_id]:13s}: {count:5d} ({count / len(df) * 100:5.2f}%)")

show_split("TRAIN", train_df)
show_split("VALIDATION", val_df)
show_split("TEST", test_df)
print("\nSố nhóm train:", train_df.component_id.nunique())
print("Số nhóm validation:", val_df.component_id.nunique())

## 6 Kiểm tra độ dài token

Phần này cho biết có bao nhiêu CV và JD dài hơn giới hạn. Nếu phần lớn JD dài hơn 125 token thì cấu hình cũ đã cắt mất nhiều thông tin.

In [ ]:
audit_tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL, use_fast=True)

def token_length_report(df, sample_size=2000):
    sample = df.sample(min(sample_size, len(df)), random_state=RANDOM_SEED)
    cv_lengths = sample["resume_text"].map(lambda x: len(audit_tokenizer(str(x), add_special_tokens=False)["input_ids"]))
    jd_lengths = sample["job_description_text"].map(lambda x: len(audit_tokenizer(str(x), add_special_tokens=False)["input_ids"]))

    report = pd.DataFrame({
        "Loại": ["CV", "JD"],
        "Trung vị": [cv_lengths.median(), jd_lengths.median()],
        "P90": [cv_lengths.quantile(0.90), jd_lengths.quantile(0.90)],
        "P95": [cv_lengths.quantile(0.95), jd_lengths.quantile(0.95)],
        "Tỷ lệ dài hơn giới hạn cũ": [(cv_lengths > 384).mean(), (jd_lengths > 125).mean()],
    })
    return report, cv_lengths, jd_lengths

length_report, cv_lengths, jd_lengths = token_length_report(train_df)
display(length_report)

plt.figure(figsize=(12, 4))
plt.subplot(1, 2, 1)
sns.histplot(cv_lengths.clip(upper=cv_lengths.quantile(0.98)), bins=40)
plt.axvline(384, color="red", linestyle="--", label="Giới hạn cũ 384")
plt.title("Độ dài CV")
plt.legend()

plt.subplot(1, 2, 2)
sns.histplot(jd_lengths.clip(upper=jd_lengths.quantile(0.98)), bins=40)
plt.axvline(125, color="red", linestyle="--", label="Giới hạn cũ 125")
plt.title("Độ dài JD")
plt.legend()
plt.tight_layout()
plt.show()

## 7 Khai báo các cấu hình cần thử

Các cấu hình chỉ khác nhau ở cách giữ nội dung và cách tính lỗi.

- `fixed_head_tail`: giữ cả phần đầu và cuối của CV/JD.
- `longest_first`: tokenizer tự cắt văn bản dài hơn.
- `weighted_ce`: tăng trọng số cho nhãn ít mẫu.
- `ce`: Cross Entropy bình thường.
- `focal`: tập trung nhiều hơn vào mẫu khó.
- `label_smoothing`: hạn chế mô hình quá tự tin.

In [ ]:
@dataclass
class ExperimentConfig:
    run_id: str
    model_name: str = BASE_MODEL
    strategy: str = "fixed_head_tail"
    jd_budget: int = 253
    loss_name: str = "weighted_ce"
    learning_rate: float = 2e-5
    epochs: int = 8
    train_batch: int = 8
    eval_batch: int = 16
    grad_accum: int = 2
    seed: int = 42

ALL_EXPERIMENTS = [
    ExperimentConfig("D0_old_input", jd_budget=125, loss_name="weighted_ce"),
    ExperimentConfig("D1_balanced_weighted", jd_budget=253, loss_name="weighted_ce"),
    ExperimentConfig("D2_longest_weighted", strategy="longest_first", loss_name="weighted_ce"),
    ExperimentConfig("D3_balanced_ce", jd_budget=253, loss_name="ce"),
    ExperimentConfig("D4_balanced_focal", jd_budget=253, loss_name="focal"),
    ExperimentConfig("D5_balanced_smoothing", jd_budget=253, loss_name="label_smoothing"),
]

if RUN_MODE == "quick":
    EXPERIMENTS = [ALL_EXPERIMENTS[i] for i in [0, 1, 3]]
elif RUN_MODE == "full":
    EXPERIMENTS = ALL_EXPERIMENTS
else:
    raise ValueError("RUN_MODE phải là 'quick' hoặc 'full'")

print("Các cấu hình sẽ chạy:")
for cfg in EXPERIMENTS:
    print("-", cfg.run_id, "|", cfg.strategy, "|", cfg.loss_name, "| JD budget =", cfg.jd_budget)

## 8 Dataset cho cặp JD và CV

JD được đặt trước CV vì yêu cầu công việc là điều kiện dùng để đánh giá ứng viên. Với chế độ cân bằng, JD được dành khoảng một nửa đầu vào thay vì chỉ 125 token.

In [ ]:
def head_tail(ids, budget):
    if len(ids) <= budget:
        return ids
    head_size = math.ceil(budget * 0.67)
    tail_size = budget - head_size
    return ids[:head_size] + ids[-tail_size:]

class CVJDCrossEncoderDataset(torch.utils.data.Dataset):
    def __init__(self, dataframe, tokenizer, config, max_length=512):
        self.df = dataframe.reset_index(drop=True)
        self.tokenizer = tokenizer
        self.config = config
        self.max_length = max_length

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        jd_text = str(row["job_description_text"])
        cv_text = str(row["resume_text"])

        if self.config.strategy == "longest_first":
            encoded = self.tokenizer(
                jd_text,
                cv_text,
                truncation="longest_first",
                max_length=self.max_length,
                padding="max_length",
                return_tensors=None,
            )
        else:
            special_tokens = self.tokenizer.num_special_tokens_to_add(pair=True)
            available = self.max_length - special_tokens
            jd_budget = min(self.config.jd_budget, available - 32)
            cv_budget = available - jd_budget

            jd_ids = self.tokenizer(jd_text, add_special_tokens=False)["input_ids"]
            cv_ids = self.tokenizer(cv_text, add_special_tokens=False)["input_ids"]
            jd_ids = head_tail(jd_ids, jd_budget)
            cv_ids = head_tail(cv_ids, cv_budget)

            encoded = self.tokenizer.prepare_for_model(
                jd_ids,
                pair_ids=cv_ids,
                add_special_tokens=True,
                max_length=self.max_length,
                padding="max_length",
                truncation=False,
                return_attention_mask=True,
            )

        item = {
            "input_ids": torch.tensor(encoded["input_ids"], dtype=torch.long),
            "attention_mask": torch.tensor(encoded["attention_mask"], dtype=torch.long),
            "labels": torch.tensor(int(row["label"]), dtype=torch.long),
        }
        if "token_type_ids" in encoded:
            item["token_type_ids"] = torch.tensor(encoded["token_type_ids"], dtype=torch.long)
        return item

## 9 Trainer và chỉ số đánh giá

Macro F1 là chỉ số chính vì nó tính đều cho cả ba nhãn. Accuracy vẫn được lưu để dễ giải thích, nhưng không dùng một mình để chọn mô hình.

In [ ]:
def compute_class_weights(df):
    counts = df["label"].value_counts().reindex([0, 1, 2], fill_value=1).values
    total = counts.sum()
    weights = total / (len(counts) * counts)
    return torch.tensor(weights, dtype=torch.float32)

CLASS_WEIGHTS = compute_class_weights(train_df)
print("Class weights:", CLASS_WEIGHTS.tolist())

class FlexibleLossTrainer(Trainer):
    def __init__(self, *args, loss_name="ce", class_weights=None, **kwargs):
        super().__init__(*args, **kwargs)
        self.loss_name = loss_name
        self.class_weights = class_weights

    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels = inputs.pop("labels")
        outputs = model(**inputs)
        logits = outputs.logits
        weights = self.class_weights.to(logits.device) if self.class_weights is not None else None

        if self.loss_name == "weighted_ce":
            loss = F.cross_entropy(logits, labels, weight=weights)
        elif self.loss_name == "focal":
            ce = F.cross_entropy(logits, labels, reduction="none")
            pt = torch.exp(-ce)
            loss = (((1 - pt) ** 2.0) * ce).mean()
        elif self.loss_name == "label_smoothing":
            loss = F.cross_entropy(logits, labels, label_smoothing=0.05)
        else:
            loss = F.cross_entropy(logits, labels)

        return (loss, outputs) if return_outputs else loss

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    per_class = f1_score(labels, preds, labels=[0, 1, 2], average=None, zero_division=0)
    return {
        "accuracy": accuracy_score(labels, preds),
        "balanced_accuracy": balanced_accuracy_score(labels, preds),
        "macro_f1": f1_score(labels, preds, labels=[0, 1, 2], average="macro", zero_division=0),
        "f1_no_fit": per_class[0],
        "f1_potential": per_class[1],
        "f1_good_fit": per_class[2],
    }

## 10 Hàm train một cấu hình

Mỗi cấu hình được train tối đa 8 epoch. Early stopping sẽ dừng nếu Macro F1 không tăng sau hai lần đánh giá. Mô hình tốt nhất của từng cấu hình được lưu riêng.

In [ ]:
def build_training_args(cfg, run_dir):
    common = dict(
        output_dir=str(run_dir / "checkpoints"),
        learning_rate=cfg.learning_rate,
        per_device_train_batch_size=cfg.train_batch,
        per_device_eval_batch_size=cfg.eval_batch,
        gradient_accumulation_steps=cfg.grad_accum,
        num_train_epochs=cfg.epochs,
        weight_decay=0.01,
        warmup_ratio=0.10,
        lr_scheduler_type="cosine",
        max_grad_norm=1.0,
        eval_strategy="epoch",
        save_strategy="epoch",
        logging_strategy="steps",
        logging_steps=50,
        load_best_model_at_end=True,
        metric_for_best_model="macro_f1",
        greater_is_better=True,
        save_total_limit=1,
        fp16=torch.cuda.is_available(),
        report_to="none",
        seed=cfg.seed,
        data_seed=cfg.seed,
        run_name=cfg.run_id,
    )
    try:
        return TrainingArguments(**common)
    except TypeError:
        common["evaluation_strategy"] = common.pop("eval_strategy")
        return TrainingArguments(**common)

def train_one_experiment(cfg):
    print("\n" + "=" * 80)
    print("ĐANG CHẠY:", cfg.run_id)
    print(asdict(cfg))
    print("=" * 80)

    seed_everything(cfg.seed)
    run_dir = OUTPUT_ROOT / cfg.run_id
    run_dir.mkdir(parents=True, exist_ok=True)

    tokenizer = AutoTokenizer.from_pretrained(cfg.model_name, use_fast=True)
    model = AutoModelForSequenceClassification.from_pretrained(
        cfg.model_name,
        num_labels=3,
        id2label=ID2LABEL,
        label2id=LABEL2ID,
    )

    train_ds = CVJDCrossEncoderDataset(train_df, tokenizer, cfg, MAX_LENGTH)
    val_ds = CVJDCrossEncoderDataset(val_df, tokenizer, cfg, MAX_LENGTH)

    trainer = FlexibleLossTrainer(
        model=model,
        args=build_training_args(cfg, run_dir),
        train_dataset=train_ds,
        eval_dataset=val_ds,
        compute_metrics=compute_metrics,
        callbacks=[EarlyStoppingCallback(early_stopping_patience=2, early_stopping_threshold=0.001)],
        loss_name=cfg.loss_name,
        class_weights=CLASS_WEIGHTS,
    )

    trainer.train()
    val_metrics = trainer.evaluate(val_ds)

    best_model_dir = run_dir / "best_model"
    trainer.save_model(str(best_model_dir))
    tokenizer.save_pretrained(str(best_model_dir))

    result = {
        **asdict(cfg),
        "best_model_dir": str(best_model_dir),
        "val_accuracy": val_metrics.get("eval_accuracy"),
        "val_balanced_accuracy": val_metrics.get("eval_balanced_accuracy"),
        "val_macro_f1": val_metrics.get("eval_macro_f1"),
        "val_f1_no_fit": val_metrics.get("eval_f1_no_fit"),
        "val_f1_potential": val_metrics.get("eval_f1_potential"),
        "val_f1_good_fit": val_metrics.get("eval_f1_good_fit"),
    }

    del trainer, model, tokenizer, train_ds, val_ds
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    return result

## 11 Chạy các thí nghiệm và chọn cấu hình tốt nhất

Cấu hình được chọn theo Macro F1 trên validation. Tập test chưa được dùng ở bước này.

In [ ]:
experiment_results = []

for cfg in EXPERIMENTS:
    try:
        experiment_results.append(train_one_experiment(cfg))
    except RuntimeError as exc:
        print("LỖI khi chạy", cfg.run_id, ":", exc)
        if "out of memory" in str(exc).lower():
            print("GPU thiếu bộ nhớ. Hãy giảm train_batch xuống 4 và tăng grad_accum lên 4.")
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

results_df = pd.DataFrame(experiment_results).sort_values(
    ["val_macro_f1", "val_f1_potential"], ascending=False
).reset_index(drop=True)

display(results_df[[
    "run_id", "strategy", "jd_budget", "loss_name", "learning_rate",
    "val_accuracy", "val_macro_f1", "val_f1_no_fit", "val_f1_potential", "val_f1_good_fit"
]])

results_path = OUTPUT_ROOT / "validation_experiment_results.csv"
results_df.to_csv(results_path, index=False)

if results_df.empty:
    raise RuntimeError("Không có cấu hình nào train thành công.")

best_row = results_df.iloc[0]
print("\nCẤU HÌNH TỐT NHẤT TRÊN VALIDATION:")
print(best_row[["run_id", "val_macro_f1", "val_f1_potential", "best_model_dir"]])

## 12 Test cuối trên 1.759 mẫu

Ô này chỉ chạy **một lần sau khi chọn xong cấu hình**. Kết quả test được xem là kết quả cuối để đưa vào báo cáo.

In [ ]:
best_cfg = next(cfg for cfg in EXPERIMENTS if cfg.run_id == best_row["run_id"])
best_model_dir = best_row["best_model_dir"]

final_tokenizer = AutoTokenizer.from_pretrained(best_model_dir, use_fast=True)
final_model = AutoModelForSequenceClassification.from_pretrained(best_model_dir)
final_test_ds = CVJDCrossEncoderDataset(test_df, final_tokenizer, best_cfg, MAX_LENGTH)

test_args = build_training_args(best_cfg, OUTPUT_ROOT / "final_test_temp")
final_trainer = FlexibleLossTrainer(
    model=final_model,
    args=test_args,
    compute_metrics=compute_metrics,
    loss_name=best_cfg.loss_name,
    class_weights=CLASS_WEIGHTS,
)

test_output = final_trainer.predict(final_test_ds)
test_logits = test_output.predictions
test_true = test_df["label"].to_numpy()
test_pred = np.argmax(test_logits, axis=-1)
test_probs = torch.softmax(torch.tensor(test_logits), dim=-1).numpy()

test_metrics = {
    "accuracy": accuracy_score(test_true, test_pred),
    "balanced_accuracy": balanced_accuracy_score(test_true, test_pred),
    "macro_f1": f1_score(test_true, test_pred, average="macro", zero_division=0),
    "f1_no_fit": f1_score(test_true, test_pred, labels=[0], average="macro", zero_division=0),
    "f1_potential": f1_score(test_true, test_pred, labels=[1], average="macro", zero_division=0),
    "f1_good_fit": f1_score(test_true, test_pred, labels=[2], average="macro", zero_division=0),
}

print("KẾT QUẢ TEST CUỐI")
for key, value in test_metrics.items():
    print(f"{key:20s}: {value:.4f}")

print("\nBáo cáo từng nhãn:")
print(classification_report(test_true, test_pred, target_names=LABEL_NAMES, digits=4, zero_division=0))

cm = confusion_matrix(test_true, test_pred, labels=[0, 1, 2])
plt.figure(figsize=(7, 6))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", xticklabels=LABEL_NAMES, yticklabels=LABEL_NAMES)
plt.xlabel("Nhãn mô hình dự đoán")
plt.ylabel("Nhãn đúng")
plt.title(f"Ma trận nhầm lẫn - {best_cfg.run_id}")
plt.tight_layout()
plt.savefig(OUTPUT_ROOT / "final_confusion_matrix.png", dpi=180, bbox_inches="tight")
plt.show()

## 13 Phân tích các lỗi lớn

Ô này lưu các mẫu đoán sai để đọc thủ công. Nên tập trung vào Potential Fit bị đoán thành No Fit hoặc Good Fit.

In [ ]:
error_df = test_df[["resume_text", "job_description_text", "label", "resume_hash", "jd_hash"]].copy()
error_df["predicted_label"] = test_pred
error_df["p_no_fit"] = test_probs[:, 0]
error_df["p_potential"] = test_probs[:, 1]
error_df["p_good_fit"] = test_probs[:, 2]
error_df["is_correct"] = error_df["label"] == error_df["predicted_label"]
error_df["true_name"] = error_df["label"].map(ID2LABEL)
error_df["predicted_name"] = error_df["predicted_label"].map(ID2LABEL)

errors_only = error_df[~error_df["is_correct"]].copy()
errors_only["error_type"] = errors_only["true_name"] + " -> " + errors_only["predicted_name"]

print("Số mẫu sai:", len(errors_only))
display(errors_only["error_type"].value_counts().to_frame("Số lượng"))

errors_only.to_csv(OUTPUT_ROOT / "final_test_errors.csv", index=False)

potential_errors = errors_only[errors_only["label"] == 1].copy()
display(potential_errors[[
    "true_name", "predicted_name", "p_no_fit", "p_potential", "p_good_fit",
    "resume_text", "job_description_text"
]].head(20))

## 14 Đánh giá xếp hạng theo JD thật

Notebook cũ tạo nhóm JD ngẫu nhiên. Phần này nhóm các mẫu có cùng nội dung JD. Chỉ nhóm có ít nhất 5 ứng viên và có từ hai mức nhãn trở lên mới được dùng.

Nếu dữ liệu không có đủ JD lặp lại, notebook sẽ thông báo rằng chưa thể đánh giá NDCG đáng tin cậy.

In [ ]:
ranking_df = test_df[["jd_hash", "label"]].copy()
ranking_df["score"] = 100 * (test_probs[:, 2] + 0.5 * test_probs[:, 1])

ndcg5_list, ndcg10_list = [], []
recall5_list, recall10_list = [], []
used_groups = 0

for jd_hash, group in ranking_df.groupby("jd_hash"):
    if len(group) < 5 or group["label"].nunique() < 2:
        continue
    used_groups += 1
    group = group.sort_values("score", ascending=False)
    y_true = group["label"].to_numpy().reshape(1, -1)
    y_score = group["score"].to_numpy().reshape(1, -1)
    ndcg5_list.append(ndcg_score(y_true, y_score, k=min(5, len(group))))
    ndcg10_list.append(ndcg_score(y_true, y_score, k=min(10, len(group))))

    good_total = max(1, int((group["label"] == 2).sum()))
    recall5_list.append(int((group.head(5)["label"] == 2).sum()) / good_total)
    recall10_list.append(int((group.head(10)["label"] == 2).sum()) / good_total)

print("Số nhóm JD đủ điều kiện:", used_groups)
if used_groups > 0:
    print("NDCG@5:", round(float(np.mean(ndcg5_list)), 4))
    print("NDCG@10:", round(float(np.mean(ndcg10_list)), 4))
    print("Recall@5:", round(float(np.mean(recall5_list)), 4))
    print("Recall@10:", round(float(np.mean(recall10_list)), 4))
else:
    print("Chưa có đủ nhóm một JD với nhiều CV để tính NDCG đáng tin cậy.")
    print("Cần tạo bộ test riêng: mỗi JD đi cùng nhiều CV ứng viên.")

## 15 Hàm dự đoán một CV và JD mới

Hàm này dùng đúng tokenizer, cách cắt văn bản và checkpoint đã được chọn. Điểm phù hợp được tính từ xác suất Good Fit và Potential Fit.

In [ ]:
def predict_cv_jd(resume_text, job_description_text):
    one_df = pd.DataFrame([{
        "resume_text": resume_text,
        "job_description_text": job_description_text,
        "label": 0,
    }])
    one_ds = CVJDCrossEncoderDataset(one_df, final_tokenizer, best_cfg, MAX_LENGTH)
    batch = one_ds[0]
    model_inputs = {
        key: value.unsqueeze(0).to(final_model.device)
        for key, value in batch.items()
        if key != "labels"
    }

    final_model.eval()
    with torch.no_grad():
        logits = final_model(**model_inputs).logits
        probs = torch.softmax(logits, dim=-1)[0].cpu().numpy()

    predicted_id = int(np.argmax(probs))
    fit_score = float(100 * (probs[2] + 0.5 * probs[1]))
    return {
        "predicted_label": ID2LABEL[predicted_id],
        "fit_score": round(fit_score, 2),
        "probability_no_fit": round(float(probs[0]), 4),
        "probability_potential_fit": round(float(probs[1]), 4),
        "probability_good_fit": round(float(probs[2]), 4),
    }

# Ví dụ sử dụng:
example_resume = "Python developer with 3 years of FastAPI, PostgreSQL, Docker and REST API experience."
example_jd = "We need a backend developer with Python, FastAPI, database and Docker experience."
print(predict_cv_jd(example_resume, example_jd))

## 16 Lưu kết quả để tải về

File ZIP chứa bảng validation, lỗi test, ma trận nhầm lẫn và toàn bộ thư mục mô hình tốt nhất.

In [ ]:
final_summary = {
    "best_run_id": best_cfg.run_id,
    "best_config": asdict(best_cfg),
    "validation_macro_f1": float(best_row["val_macro_f1"]),
    "test_metrics": {key: float(value) for key, value in test_metrics.items()},
}

with open(OUTPUT_ROOT / "final_summary.json", "w", encoding="utf-8") as f:
    json.dump(final_summary, f, ensure_ascii=False, indent=2)

zip_path = shutil.make_archive("/content/CV_JD_OPTIMIZED_RESULTS", "zip", OUTPUT_ROOT)
print("Đã tạo:", zip_path)

from google.colab import files
files.download(zip_path)

## Ghi chú khi đọc kết quả

- Không chọn mô hình chỉ vì Accuracy cao.
- Ưu tiên Macro F1 và F1 của Potential Fit.
- Nếu validation vẫn cao nhưng test giảm mạnh, quay lại kiểm tra dữ liệu và nhãn; không tiếp tục tăng epoch.
- Không đưa dữ liệu test vào train.
- Kết quả AI chỉ nên hỗ trợ sắp xếp hồ sơ. Nhà tuyển dụng vẫn cần đọc CV gốc trước khi quyết định.